# AlpCAN CT Pipeline: Nodül Karakterizasyon Eğitimi

**Amaç:** LIDC-IDRI veri seti üzerinde nodül karakterizasyon modeli eğitmek.
AlpCAN CT Pipeline Agent 3 (Nodül Karakterizasyon) için ağırlık üretimi.

**İçindekiler:**
1. LIDC-IDRI veri seti yükleme ve nodül patch çıkarımı
2. Etiket hazırlama: Malignite skoru (4 radyolog konsensüs)
3. 2D ResNet-50 + CBAM modeli
4. Eğitim (Binary CE + Focal Loss, 40 epoch)
5. AUC-ROC ve confusion matrix
6. Grad-CAM görselleştirme
7. Lung-RADS eşleme
8. Sonuçları kaydetme + ağırlık export

**Model:** ResNet-50 + CBAM (Channel & Spatial Attention)
**GPU:** Kaggle T4 16GB
**Dataset:** `zhangweiled/lidcidri` — LIDC-IDRI kesilmiş nodül dilimleri

**Not:** LIDC-IDRI kesilmiş dilim formatı 2D'dir. Tam 3D karakterizasyon gelecek fazda
tam volumetrik BT serileri ile yapılacaktır. Bu notebook 2D slice-level baseline oluşturur.

In [ ]:
!pip install -q grad-cam albumentations

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report

print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Bellek: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

---
## 1. LIDC-IDRI Veri Seti ve Nodül Patch Çıkarımı

In [ ]:
INPUT_DIR = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# LIDC-IDRI dataset
DATA_DIR = Path("/kaggle/input/lidcidri/LIDC-IDRI-slices")
if not DATA_DIR.exists():
    for candidate in INPUT_DIR.rglob("LIDC-IDRI-0001"):
        DATA_DIR = candidate.parent
        break
    if not DATA_DIR.exists():
        raise FileNotFoundError("LIDC-IDRI dataset bulunamadi!")

print(f"Dataset: {DATA_DIR}")

patient_dirs = sorted([d for d in DATA_DIR.iterdir() if d.is_dir()])
print(f"Toplam hasta: {len(patient_dirs)}")

In [ ]:
# Her nodul icin: orta dilim + maske boyutu + annotator sayisi
# Malignite etiketi: maskelerin varligini/boyutunu proxy olarak kullan
# (Gercek LIDC-IDRI XML annotasyonlari bu dataset'te yok,
#  maske boyutu ve annotator uyumu ile malignite tahmini yapilir)

nodule_data = []

for pdir in patient_dirs:
    nodule_dirs = sorted([d for d in pdir.iterdir() if d.is_dir()])
    for ndir in nodule_dirs:
        img_dir = ndir / "images"
        mask_dirs = sorted([d for d in ndir.iterdir() if d.is_dir() and d.name.startswith("mask")])
        
        if not img_dir.exists() or not mask_dirs:
            continue
        
        img_files = sorted(img_dir.glob("*.png"))
        if not img_files:
            continue
        
        # Orta dilimi sec (nodul en buyuk oldugu dilim)
        mid_idx = len(img_files) // 2
        mid_img = img_files[mid_idx]
        
        # Tum annotator maskelerinden boyut hesapla
        mask_areas = []
        mask_agreement = 0
        combined_mask = None
        
        for mdir in mask_dirs:
            mask_file = mdir / mid_img.name
            if mask_file.exists():
                m = np.array(Image.open(mask_file).convert('L'))
                area = (m > 0).sum()
                mask_areas.append(area)
                if combined_mask is None:
                    combined_mask = (m > 0).astype(np.float32)
                else:
                    combined_mask += (m > 0).astype(np.float32)
        
        if not mask_areas or combined_mask is None:
            continue
        
        avg_area = np.mean(mask_areas)
        n_annotators = len(mask_areas)
        consensus_area = ((combined_mask / n_annotators) >= 0.5).sum()
        
        # Nodul boyutuna gore basit malignite proxy:
        # Buyuk noduller (>= 100 piksel konsensus alani) daha suphe
        # Bu basit bir heuristik — gercek malignite skorlari XML'de
        # Ancak boyut Lung-RADS'in temel kriteridir
        
        # Nodul capini piksel olarak hesapla
        diameter_px = np.sqrt(consensus_area / np.pi) * 2 if consensus_area > 0 else 0
        
        nodule_data.append({
            'patient_id': pdir.name,
            'nodule': ndir.name,
            'image_path': str(mid_img),
            'mask_dirs': [str(m) for m in mask_dirs],
            'n_annotators': n_annotators,
            'avg_mask_area': avg_area,
            'consensus_area': consensus_area,
            'diameter_px': diameter_px,
            'n_slices': len(img_files),
        })

df = pd.DataFrame(nodule_data)
print(f"\nToplam nodul: {len(df)}")
print(f"Toplam hasta: {df['patient_id'].nunique()}")
print(f"\nNodul cap (piksel) istatistikleri:")
print(df['diameter_px'].describe())

---
## 2. Etiket Hazırlama — Boyut Bazlı Sınıflandırma

In [ ]:
# Lung-RADS benzeri boyut siniflandirmasi (piksel bazli proxy)
# Gercek mm olcusu icin DICOM spacing bilgisi gerekli
# Bu dataset'te spacing bilgisi yok, piksel boyutunu kullaniyoruz

# Boyut kategorizasyonu: kucuk (<= medyan) vs buyuk (> medyan)
# Bu binary siniflandirma "suphe" tahmini icin baseline
median_diameter = df['diameter_px'].median()
print(f"Medyan nodul capi: {median_diameter:.1f} piksel")

# Ek olarak: annotator uyumu da onemli
# 4/4 annotator isaretlemis = daha belirgin nodul
# 1/4 annotator = belirsiz/kucuk

# Suspicious etiketi: buyuk nodul VE yuksek annotator uyumu
df['size_category'] = pd.cut(
    df['diameter_px'],
    bins=[0, median_diameter * 0.5, median_diameter, median_diameter * 2, float('inf')],
    labels=['cok_kucuk', 'kucuk', 'orta', 'buyuk']
)

# Binary: kucuk (benign-like) vs buyuk (suspicious)
df['suspicious'] = ((df['diameter_px'] > median_diameter) & (df['n_annotators'] >= 3)).astype(int)

# Multi-class: 4 sinif
def assign_risk_class(row):
    d = row['diameter_px']
    n = row['n_annotators']
    if d <= median_diameter * 0.5:
        return 0  # Lung-RADS 1 benzeri
    elif d <= median_diameter:
        return 1  # Lung-RADS 2 benzeri
    elif d <= median_diameter * 2:
        return 2  # Lung-RADS 3 benzeri
    else:
        return 3  # Lung-RADS 4 benzeri

df['risk_class'] = df.apply(assign_risk_class, axis=1)

print(f"\nBinary siniflandirma (suspicious):")
print(df['suspicious'].value_counts())
print(f"\nRisk sinifi dagilimi:")
print(df['risk_class'].value_counts().sort_index())
print(f"\nBoyut kategorisi:")
print(df['size_category'].value_counts())

In [ ]:
# Etiket dagilimi gorsellestir
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Nodul cap dagilimi
axes[0].hist(df['diameter_px'], bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(x=median_diameter, color='red', linestyle='--', label=f'Medyan: {median_diameter:.1f}')
axes[0].set_title('Nodul Cap Dagilimi', fontweight='bold')
axes[0].set_xlabel('Cap (piksel)')
axes[0].set_ylabel('Sayi')
axes[0].legend()

# Annotator uyumu
axes[1].bar(df['n_annotators'].value_counts().sort_index().index,
            df['n_annotators'].value_counts().sort_index().values,
            color='coral', edgecolor='black')
axes[1].set_title('Annotator Uyumu', fontweight='bold')
axes[1].set_xlabel('Annotator Sayisi')
axes[1].set_ylabel('Nodul Sayisi')

# Risk sinifi
risk_counts = df['risk_class'].value_counts().sort_index()
colors = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c']
risk_labels = ['Dusuk Risk', 'Orta-Dusuk', 'Orta-Yuksek', 'Yuksek Risk']
axes[2].bar(risk_labels, [risk_counts.get(i, 0) for i in range(4)],
            color=colors, edgecolor='black')
axes[2].set_title('Risk Sinifi Dagilimi', fontweight='bold')
axes[2].set_ylabel('Nodul Sayisi')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'ct_char_label_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Eğitim/Doğrulama Bölmesi ve Dataset

In [ ]:
# Hasta bazli bolme
np.random.seed(42)
patients = df['patient_id'].unique()
np.random.shuffle(patients)

split_idx = int(len(patients) * 0.8)
train_patients = set(patients[:split_idx])
val_patients = set(patients[split_idx:])

train_df = df[df['patient_id'].isin(train_patients)].reset_index(drop=True)
val_df = df[df['patient_id'].isin(val_patients)].reset_index(drop=True)

print(f"Train: {len(train_df)} nodul ({len(train_patients)} hasta)")
print(f"Val:   {len(val_df)} nodul ({len(val_patients)} hasta)")
print(f"\nTrain suspicious orani: {train_df['suspicious'].mean():.2%}")
print(f"Val suspicious orani:   {val_df['suspicious'].mean():.2%}")

In [ ]:
PATCH_SIZE = 128

train_transform = A.Compose([
    A.Resize(PATCH_SIZE, PATCH_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=45, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.2),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(PATCH_SIZE, PATCH_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])


class NoduleCharDataset(Dataset):
    """Nodul karakterizasyon dataset'i.
    
    Her nodul icin orta dilim goruntusunu yukler.
    Maskeleri kullanarak nodul bolgesine crop yapar.
    """
    def __init__(self, dataframe, transform=None, crop_to_nodule=True):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.crop_to_nodule = crop_to_nodule
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Goruntu oku
        img = np.array(Image.open(row['image_path']).convert('L'))
        
        if self.crop_to_nodule and row['mask_dirs']:
            # Konsensus maske ile nodul bolgesini bul
            combined = np.zeros(img.shape, dtype=np.float32)
            n = 0
            for mdir in row['mask_dirs']:
                mask_file = Path(mdir) / Path(row['image_path']).name
                if mask_file.exists():
                    m = np.array(Image.open(mask_file).convert('L'))
                    combined += (m > 0).astype(np.float32)
                    n += 1
            
            if n > 0:
                consensus = (combined / n) >= 0.5
                if consensus.any():
                    # Nodul merkezi ve genisletilmis crop
                    rows_with = np.where(np.any(consensus, axis=1))[0]
                    cols_with = np.where(np.any(consensus, axis=0))[0]
                    
                    cy = (rows_with[0] + rows_with[-1]) // 2
                    cx = (cols_with[0] + cols_with[-1]) // 2
                    h = rows_with[-1] - rows_with[0]
                    w = cols_with[-1] - cols_with[0]
                    
                    # 2x nodul boyutu kadar context ekle
                    margin = max(h, w)
                    half = max(margin, 32)  # minimum 32 piksel
                    
                    y1 = max(0, cy - half)
                    y2 = min(img.shape[0], cy + half)
                    x1 = max(0, cx - half)
                    x2 = min(img.shape[1], cx + half)
                    
                    img = img[y1:y2, x1:x2]
        
        # 3 kanala cevir
        img_rgb = np.stack([img, img, img], axis=-1)
        
        if self.transform:
            augmented = self.transform(image=img_rgb)
            img_tensor = augmented['image']
        else:
            img_tensor = torch.from_numpy(img_rgb.transpose(2, 0, 1)).float() / 255.0
        
        # Etiketler
        suspicious = torch.tensor(row['suspicious'], dtype=torch.float32)
        risk_class = torch.tensor(row['risk_class'], dtype=torch.long)
        
        return img_tensor, suspicious, risk_class


train_dataset = NoduleCharDataset(train_df, transform=train_transform)
val_dataset = NoduleCharDataset(val_df, transform=val_transform)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")
print(f"Patch boyutu: {PATCH_SIZE}x{PATCH_SIZE}")

img_sample, susp, risk = train_dataset[0]
print(f"Goruntu shape: {img_sample.shape}, Suspicious: {susp.item()}, Risk: {risk.item()}")

---
## 4. Model — ResNet-50 + CBAM

In [ ]:
class ChannelAttention(nn.Module):
    """CBAM Channel Attention."""
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
        )
    
    def forward(self, x):
        b, c, _, _ = x.size()
        avg_out = self.fc(self.avg_pool(x).view(b, c))
        max_out = self.fc(self.max_pool(x).view(b, c))
        attention = torch.sigmoid(avg_out + max_out).view(b, c, 1, 1)
        return x * attention


class SpatialAttention(nn.Module):
    """CBAM Spatial Attention."""
    def __init__(self, kernel_size=7):
        super().__init__()
        padding = kernel_size // 2
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
    
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        concat = torch.cat([avg_out, max_out], dim=1)
        attention = torch.sigmoid(self.conv(concat))
        return x * attention


class CBAM(nn.Module):
    """Convolutional Block Attention Module."""
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_att = ChannelAttention(channels, reduction)
        self.spatial_att = SpatialAttention(kernel_size)
    
    def forward(self, x):
        x = self.channel_att(x)
        x = self.spatial_att(x)
        return x


class ResNet50CBAM(nn.Module):
    """ResNet-50 + CBAM nodul karakterizasyon modeli.
    
    Cikislar:
    - suspicious: Binary (0/1) — nodul supheli mi?
    - risk_class: 4-sinif (0-3) — risk seviyesi
    """
    def __init__(self, n_risk_classes=4):
        super().__init__()
        # ResNet-50 backbone
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.features = nn.Sequential(*list(resnet.children())[:-2])  # Son FC ve avgpool haric
        
        # CBAM attention
        self.cbam = CBAM(2048, reduction=16)
        
        # Global average pooling
        self.gap = nn.AdaptiveAvgPool2d(1)
        
        # Classification heads
        self.dropout = nn.Dropout(0.3)
        self.fc_suspicious = nn.Linear(2048, 1)  # Binary
        self.fc_risk = nn.Linear(2048, n_risk_classes)  # Multi-class
    
    def forward(self, x):
        features = self.features(x)  # (B, 2048, H/32, W/32)
        features = self.cbam(features)
        pooled = self.gap(features).flatten(1)  # (B, 2048)
        pooled = self.dropout(pooled)
        
        suspicious = self.fc_suspicious(pooled).squeeze(-1)  # (B,)
        risk = self.fc_risk(pooled)  # (B, 4)
        
        return suspicious, risk, features  # features for Grad-CAM


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ResNet50CBAM(n_risk_classes=4).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model: ResNet-50 + CBAM")
print(f"Parametre: {n_params:,} ({n_params/1e6:.1f}M)")
print(f"Device: {device}")
print(f"Cikislar: suspicious (binary), risk_class (4-sinif)")

---
## 5. Loss ve Optimizer

In [ ]:
class FocalLoss(nn.Module):
    """Focal Loss — class imbalance icin."""
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    
    def forward(self, pred, target):
        bce = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
        pt = torch.exp(-bce)
        focal = self.alpha * (1 - pt) ** self.gamma * bce
        return focal.mean()


# Class weights hesapla (risk sinifi imbalance icin)
risk_counts = train_df['risk_class'].value_counts().sort_index()
risk_weights = 1.0 / (risk_counts.values.astype(np.float32) + 1)
risk_weights = risk_weights / risk_weights.sum() * len(risk_weights)
risk_weights_tensor = torch.tensor(risk_weights, dtype=torch.float32).to(device)
print(f"Risk class weights: {risk_weights}")

criterion_suspicious = FocalLoss(alpha=0.25, gamma=2.0)
criterion_risk = nn.CrossEntropyLoss(weight=risk_weights_tensor)

optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40, eta_min=1e-6)

print("Suspicious loss: Focal (alpha=0.25, gamma=2)")
print("Risk loss: Weighted CrossEntropy")
print("Optimizer: AdamW (lr=1e-4)")

---
## 6. Eğitim

In [ ]:
NUM_EPOCHS = 40
best_val_auc = 0.0
history = {'train_loss': [], 'val_loss': [], 'val_auc': [], 'val_risk_acc': [], 'lr': []}
scaler = torch.amp.GradScaler('cuda')

SUSPICIOUS_WEIGHT = 1.0
RISK_WEIGHT = 0.5

print(f"Egitim basliyor — {NUM_EPOCHS} epoch")
print(f"Loss agirliklari: suspicious={SUSPICIOUS_WEIGHT}, risk={RISK_WEIGHT}")
print("=" * 70)

start_time = time.time()

for epoch in range(NUM_EPOCHS):
    # --- Train ---
    model.train()
    train_loss = 0.0
    n_train = 0
    
    for images, susp_labels, risk_labels in train_loader:
        images = images.to(device)
        susp_labels = susp_labels.to(device)
        risk_labels = risk_labels.to(device)
        
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            susp_pred, risk_pred, _ = model(images)
            loss_susp = criterion_suspicious(susp_pred, susp_labels)
            loss_risk = criterion_risk(risk_pred, risk_labels)
            loss = SUSPICIOUS_WEIGHT * loss_susp + RISK_WEIGHT * loss_risk
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        train_loss += loss.item() * images.size(0)
        n_train += images.size(0)
    
    train_loss /= n_train
    
    # --- Validation ---
    model.eval()
    val_loss = 0.0
    all_susp_preds = []
    all_susp_labels = []
    all_risk_preds = []
    all_risk_labels = []
    n_val = 0
    
    with torch.no_grad():
        for images, susp_labels, risk_labels in val_loader:
            images = images.to(device)
            susp_labels = susp_labels.to(device)
            risk_labels = risk_labels.to(device)
            
            with torch.amp.autocast('cuda'):
                susp_pred, risk_pred, _ = model(images)
                loss_susp = criterion_suspicious(susp_pred, susp_labels)
                loss_risk = criterion_risk(risk_pred, risk_labels)
                loss = SUSPICIOUS_WEIGHT * loss_susp + RISK_WEIGHT * loss_risk
            
            val_loss += loss.item() * images.size(0)
            all_susp_preds.extend(torch.sigmoid(susp_pred).cpu().numpy())
            all_susp_labels.extend(susp_labels.cpu().numpy())
            all_risk_preds.extend(risk_pred.argmax(dim=1).cpu().numpy())
            all_risk_labels.extend(risk_labels.cpu().numpy())
            n_val += images.size(0)
    
    val_loss /= n_val
    
    # Metrikler
    all_susp_preds = np.array(all_susp_preds)
    all_susp_labels = np.array(all_susp_labels)
    all_risk_preds = np.array(all_risk_preds)
    all_risk_labels = np.array(all_risk_labels)
    
    if len(np.unique(all_susp_labels)) > 1:
        val_auc = roc_auc_score(all_susp_labels, all_susp_preds)
    else:
        val_auc = 0.0
    
    risk_acc = (all_risk_preds == all_risk_labels).mean()
    
    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_auc'].append(val_auc)
    history['val_risk_acc'].append(risk_acc)
    history['lr'].append(current_lr)
    
    # Best model
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_auc': best_val_auc,
            'val_risk_acc': risk_acc,
        }, OUTPUT_DIR / 'ct_char_best_model.pth')
        marker = ' << BEST'
    else:
        marker = ''
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        elapsed = time.time() - start_time
        print(f"Epoch {epoch+1:>3d}/{NUM_EPOCHS} | "
              f"Train: {train_loss:.4f} | "
              f"Val: {val_loss:.4f} | "
              f"AUC: {val_auc:.4f} | "
              f"Risk Acc: {risk_acc:.4f} | "
              f"{elapsed/60:.1f}m{marker}")

total_time = time.time() - start_time
print(f"\nEgitim tamamlandi! Sure: {total_time/60:.1f} dk")
print(f"En iyi AUC: {best_val_auc:.4f}")

---
## 7. Eğitim Grafikleri ve Değerlendirme

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0].plot(history['val_loss'], label='Val', linewidth=2)
axes[0].set_title('Loss', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['val_auc'], label='Suspicious AUC', linewidth=2, color='#e74c3c')
axes[1].plot(history['val_risk_acc'], label='Risk Accuracy', linewidth=2, color='#3498db')
axes[1].set_title('Validasyon Metrikleri', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 1)

axes[2].plot(history['lr'], linewidth=2, color='#2ecc71')
axes[2].set_title('Learning Rate', fontweight='bold')
axes[2].set_xlabel('Epoch')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'ct_char_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# En iyi modeli yukle ve detayli degerlendirme
checkpoint = torch.load(OUTPUT_DIR / 'ct_char_best_model.pth', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"Best model yuklendi (Epoch {checkpoint['epoch']+1}, AUC={checkpoint['val_auc']:.4f})")

# Tam validasyon tahminleri
all_susp_preds = []
all_susp_labels = []
all_risk_preds = []
all_risk_labels = []

with torch.no_grad():
    for images, susp_labels, risk_labels in val_loader:
        images = images.to(device)
        with torch.amp.autocast('cuda'):
            susp_pred, risk_pred, _ = model(images)
        all_susp_preds.extend(torch.sigmoid(susp_pred).cpu().numpy())
        all_susp_labels.extend(susp_labels.numpy())
        all_risk_preds.extend(risk_pred.argmax(dim=1).cpu().numpy())
        all_risk_labels.extend(risk_labels.numpy())

all_susp_preds = np.array(all_susp_preds)
all_susp_labels = np.array(all_susp_labels)
all_risk_preds = np.array(all_risk_preds)
all_risk_labels = np.array(all_risk_labels)

# ROC egrisi
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

if len(np.unique(all_susp_labels)) > 1:
    fpr, tpr, thresholds = roc_curve(all_susp_labels, all_susp_preds)
    auc_val = roc_auc_score(all_susp_labels, all_susp_preds)
    
    axes[0].plot(fpr, tpr, linewidth=2, color='#e74c3c', label=f'AUC = {auc_val:.4f}')
    axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3)
    axes[0].set_xlabel('False Positive Rate')
    axes[0].set_ylabel('True Positive Rate')
    axes[0].set_title('ROC Egrisi — Suspicious Detection', fontweight='bold')
    axes[0].legend(fontsize=12)
    axes[0].grid(True, alpha=0.2)

# Confusion matrix
cm = confusion_matrix(all_risk_labels, all_risk_preds)
risk_names = ['Dusuk', 'Orta-D', 'Orta-Y', 'Yuksek']
im = axes[1].imshow(cm, interpolation='nearest', cmap='Blues')
axes[1].set_title('Risk Sinifi Confusion Matrix', fontweight='bold')
axes[1].set_xlabel('Tahmin')
axes[1].set_ylabel('Gercek')
axes[1].set_xticks(range(len(risk_names)))
axes[1].set_yticks(range(len(risk_names)))
axes[1].set_xticklabels(risk_names)
axes[1].set_yticklabels(risk_names)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        axes[1].text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > cm.max() / 2 else 'black')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'ct_char_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nRisk Sinifi Classification Report:")
print(classification_report(all_risk_labels, all_risk_preds, target_names=risk_names))

---
## 8. Grad-CAM Görselleştirme

In [ ]:
# Basit Grad-CAM implementasyonu
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        target_layer.register_forward_hook(self._forward_hook)
        target_layer.register_full_backward_hook(self._backward_hook)
    
    def _forward_hook(self, module, input, output):
        self.activations = output.detach()
    
    def _backward_hook(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()
    
    def generate(self, input_tensor, target_class=None):
        self.model.eval()
        output, _, _ = self.model(input_tensor)
        
        if target_class is None:
            target = output
        else:
            target = output[:, target_class] if output.dim() > 1 else output
        
        self.model.zero_grad()
        target.sum().backward()
        
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=input_tensor.shape[2:], mode='bilinear', align_corners=False)
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        return cam.cpu().numpy()[0, 0]


# Son convolutional layer'i bul
target_layer = model.features[-1][-1].conv3  # ResNet50 son bottleneck'in son conv'u
grad_cam = GradCAM(model, target_layer)

# 8 ornek icin Grad-CAM gorsellestir
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Grad-CAM — Model Dikkat Haritasi', fontsize=14)

indices = np.linspace(0, len(val_dataset) - 1, 8, dtype=int)

for col_idx, (ax, idx) in enumerate(zip(axes.flat, indices)):
    img, susp_label, risk_label = val_dataset[idx]
    img_gpu = img.unsqueeze(0).to(device).requires_grad_(True)
    
    cam = grad_cam.generate(img_gpu)
    
    # Goruntu denormalize
    img_np = img.numpy().transpose(1, 2, 0)
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img_np = (img_np * std + mean).clip(0, 1)
    
    ax.imshow(img_np[:, :, 0], cmap='gray')
    ax.imshow(cam, cmap='jet', alpha=0.4)
    
    with torch.no_grad():
        susp_pred, risk_pred, _ = model(img_gpu)
        susp_score = torch.sigmoid(susp_pred).item()
    
    risk_names_short = ['DR', 'OD', 'OY', 'YR']
    ax.set_title(f'Susp: {susp_score:.2f} | R: {risk_names_short[risk_label]}', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'ct_char_gradcam.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Lung-RADS Eşleme ve Final Özet

In [ ]:
# Risk sinifini Lung-RADS kategorisine esle
RISK_TO_LUNGRADS = {
    0: '1',    # Dusuk risk -> Lung-RADS 1 (Negatif)
    1: '2',    # Orta-Dusuk -> Lung-RADS 2 (Benign)
    2: '3',    # Orta-Yuksek -> Lung-RADS 3 (Muhtemelen Benign)
    3: '4A',   # Yuksek risk -> Lung-RADS 4A (Suphe)
}

print("Risk Sinifi -> Lung-RADS Esleme:")
for risk, lr in RISK_TO_LUNGRADS.items():
    n_val = (all_risk_labels == risk).sum()
    n_pred = (all_risk_preds == risk).sum()
    print(f"  Risk {risk} -> Lung-RADS {lr}: {n_val} gercek, {n_pred} tahmin")

# Sonuclari kaydet
results_df = val_df.copy()
results_df['susp_pred'] = all_susp_preds
results_df['risk_pred'] = all_risk_preds
results_df['lung_rads_pred'] = [RISK_TO_LUNGRADS[r] for r in all_risk_preds]
results_df.to_csv(OUTPUT_DIR / 'ct_char_predictions.csv', index=False)

# Metrikler
metrics = {
    'suspicious_auc': roc_auc_score(all_susp_labels, all_susp_preds) if len(np.unique(all_susp_labels)) > 1 else 0,
    'risk_accuracy': (all_risk_preds == all_risk_labels).mean(),
}
pd.DataFrame([metrics]).to_csv(OUTPUT_DIR / 'ct_char_metrics.csv', index=False)

# History
pd.DataFrame(history).to_csv(OUTPUT_DIR / 'ct_char_training_history.csv', index=False)

print(f"\nKaydedilen dosyalar:")
for f in sorted(OUTPUT_DIR.glob('ct_char_*')):
    print(f"  {f.name} ({f.stat().st_size / 1024:.0f} KB)")

In [ ]:
print("\n" + "=" * 65)
print("AlpCAN CT Pipeline — Nodul Karakterizasyon Egitimi OZET")
print("=" * 65)

print(f"\n--- Dataset ---")
print(f"  LIDC-IDRI (zhangweiled/lidcidri)")
print(f"  Train: {len(train_df)} nodul ({len(train_patients)} hasta)")
print(f"  Val:   {len(val_df)} nodul ({len(val_patients)} hasta)")
print(f"  Etiket: Boyut bazli risk siniflandirma (4 sinif)")

print(f"\n--- Model ---")
print(f"  Mimari: ResNet-50 + CBAM")
print(f"  Giris: {PATCH_SIZE}x{PATCH_SIZE}x3 (nodul patch, context dahil)")
print(f"  Parametre: {n_params:,} ({n_params/1e6:.1f}M)")
print(f"  Loss: Focal (suspicious) + Weighted CE (risk)")
print(f"  Epoch: {NUM_EPOCHS}")

print(f"\n--- Performans ---")
print(f"  Suspicious AUC: {metrics['suspicious_auc']:.4f}")
print(f"  Risk Accuracy:  {metrics['risk_accuracy']:.4f}")

print(f"\n--- Lung-RADS Esleme ---")
for risk, lr in RISK_TO_LUNGRADS.items():
    print(f"  Risk {risk} -> Lung-RADS {lr}")

print(f"\n--- Sonraki Adimlar ---")
print(f"  1. Model agirliklarini AlpCAN backend'e entegre et")
print(f"  2. characterization_inference.py guncelle")
print(f"  3. Nodul segmentasyon + karakterizasyon pipeline birlestir")
print(f"  4. Gercek LIDC-IDRI XML annotasyonlarindan malignite etiketleri")
print(f"  5. 3D volumetrik analiz (tam BT serisi)")
print("=" * 65)